# Evaluating Clusters & Choosing k

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/clustering/03-evaluating-clusters

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Intuition — how do you know the clusters are any good?

Clustering is **unsupervised**, so there's usually no answer key. Two families of metrics fill the
gap. **Internal** metrics judge the geometry of the clustering itself: **inertia** (within-cluster
spread — the elbow plot's y-axis) and the **silhouette** (how much closer each point is to its own
cluster than the next-nearest, from −1 to +1). **External** metrics (like the adjusted Rand index)
compare against ground-truth labels *when you have them*. The crucial catch — and the theme of this
notebook — is that internal metrics measure **compactness, not correctness**, so they quietly reward
the same blob-shaped bias k-means has, and can rate a *wrong* clustering higher than the true one.

## K-Means + the two classic diagnostics

Three true blobs; we let k range from 2 to 8 and see which k the metrics point at.

In [ ]:
def kmeans(X, k, iters=50):
    C = X[np.random.choice(len(X), k, replace=False)]
    for _ in range(iters):
        lbl = np.argmin(((X[:, None] - C[None]) ** 2).sum(-1), axis=1)
        C = np.array([X[lbl == j].mean(0) if (lbl == j).any() else C[j] for j in range(k)])
    inertia = ((X - C[lbl]) ** 2).sum()
    return lbl, C, inertia

blobs = [np.random.randn(60, 2) * 0.7 + c for c in [(-3, 0), (3, 2), (1, -3)]]
X = np.vstack(blobs)

**What to notice:** three well-separated blobs — the easy case where every diagnostic agrees. The
`kmeans` helper returns the labels, centroids, and **inertia** (total within-cluster squared distance),
the quantity the elbow plot tracks.

## Silhouette, from its definition

In [ ]:
def silhouette(X, lbl):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    s = np.zeros(len(X))
    for i in range(len(X)):
        own = lbl == lbl[i]; own[i] = False
        a = D[i, own].mean() if own.any() else 0
        b = min(D[i, lbl == j].mean() for j in set(lbl) if j != lbl[i])
        s[i] = (b - a) / max(a, b)
    return s

ks = range(2, 9); inertias, sils = [], []
for k in ks:
    lbl, C, inertia = kmeans(X, k)
    inertias.append(inertia); sils.append(silhouette(X, lbl).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(ks), inertias, 'o-', color='#6366f1'); axes[0].set_title('inertia (elbow)')
axes[1].plot(list(ks), sils, 'o-', color='#14b8a6'); axes[1].set_title('mean silhouette')
for ax in axes: ax.set_xlabel('k'); ax.axvline(3, color='#f43f5e', ls=':', lw=1)
plt.tight_layout(); plt.show()
# inertia bends at k=3; silhouette peaks at k=3 — both agree here

**What to notice:** the **elbow** in inertia bends at `k=3` and the mean **silhouette** peaks at
`k=3` — both correctly identify the three blobs. When clusters are clean and round, the internal
metrics are trustworthy and mutually confirming.

## The library way — validate the silhouette against `sklearn`

Our from-scratch silhouette should match `sklearn.metrics.silhouette_score` exactly. The cell checks
it on the three-blob data.

In [ ]:
from sklearn.metrics import silhouette_score

lbl, C, _ = kmeans(X, 3)
ours = silhouette(X, lbl).mean()
sk = silhouette_score(X, lbl)
print(f'our silhouette: {ours:.4f} | sklearn: {sk:.4f}')
assert np.isclose(ours, sk), "our silhouette must match sklearn"
print('our silhouette == sklearn.metrics.silhouette_score ✓')

**What to notice:** exact match — our definition-based silhouette is `sklearn`'s. Which makes the
next result all the more important: the metric is correct, but *what it measures* (compactness) isn't
the same as *what you want* (the right clustering).

## When the metrics lie: two moons

In [ ]:
t = np.linspace(0, np.pi, 100)
moons = np.vstack([np.c_[np.cos(t), np.sin(t)],
                   np.c_[1 - np.cos(t), 0.5 - np.sin(t)]]) + 0.07 * np.random.randn(200, 2)
true_lbl = np.array([0] * 100 + [1] * 100)

km_lbl, _, _ = kmeans(moons, 2)
print('silhouette of TRUE moon labels :', round(silhouette(moons, true_lbl).mean(), 3))
print('silhouette of K-Means labels   :', round(silhouette(moons, km_lbl).mean(), 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, l, name in [(axes[0], true_lbl, 'true clusters'), (axes[1], km_lbl, 'K-Means k=2')]:
    ax.scatter(*moons.T, c=np.where(l == 0, '#6366f1', '#14b8a6'), s=12); ax.set_title(name)
plt.tight_layout(); plt.show()
# K-Means' wrong vertical split can SCORE HIGHER than the true crescents:
# silhouette rewards compact round blobs — the same bias K-Means has

**What to notice:** the twist — on two moons, **k-means's wrong vertical split scores a *higher*
silhouette than the true crescent labels**. The silhouette rewards compact, round groups, which is
exactly k-means's bias, so it happily certifies the wrong answer. An internal metric measuring
compactness can't detect a clustering that's compact-but-wrong.

## Gotchas & tradeoffs

- **Internal metrics measure geometry, not truth.** Silhouette/inertia reward compact spherical
  clusters, so they penalize *correct* non-convex clusters (DBSCAN's moons) and can crown wrong ones.
- **Use external metrics when you have labels.** ARI / NMI compare to ground truth and aren't fooled by
  shape — the honest check when a reference exists.
- **The elbow is subjective.** The bend is often ambiguous; combine it with silhouette and domain
  knowledge, and consider cluster **stability** across reruns/subsamples.
- **No metric replaces looking at the data.** Always visualize (or inspect) the clusters — a number
  can hide a nonsensical grouping.

In [ ]:
# Internal metric (silhouette) vs external metric (ARI) on the moons
from sklearn.metrics import adjusted_rand_score
print(f'silhouette: k-means split = {silhouette(moons, km_lbl).mean():.3f}  '
      f'(looks as good/better than the truth)')
print(f'ARI vs ground truth: k-means = {adjusted_rand_score(true_lbl, km_lbl):.3f}  '
      f'(near 0 -> actually wrong!)')
print('\n-> the internal metric is fooled; the external metric (with labels) tells the truth')

**What to notice:** silhouette rates the k-means split as good, but the **ARI against ground truth
is near 0** — the clustering is actually wrong. This is the core lesson of cluster evaluation: an
internal score alone can mislead; when you have labels, trust an **external** metric, and always
sanity-check by eye.

## Key takeaways

- **Internal** metrics (inertia/elbow, silhouette) judge geometry without labels; **external** metrics
  (ARI/NMI) compare to ground truth.
- Elbow + silhouette **agree on clean blobs** and are trustworthy there.
- But internal metrics reward **compactness**, sharing k-means's bias — they can score a *wrong*
  clustering above the true one (the moons).
- Use **external metrics when labels exist**, check **stability**, and **always visualize** — no single
  number certifies a clustering.

**Next:** the [course quiz](https://ml-viz-ruby.vercel.app/courses/clustering/04-quiz).

**Try it:** write the stability check — run `kmeans` 20 times on random 80% subsets and count how often each pair of points co-clusters. At k=3 on the blobs, co-clustering rates are near 0 or 1 (stable); at k=5 they smear (unstable).

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The silhouette formula

For one point with mean intra-cluster distance $a$ and mean nearest-other-cluster distance $b$:

$$s = \frac{b - a}{\max(a, b)}$$

Implement it and verify the landmarks: well-placed points push toward $+1$, boundary points give $0$, misassigned points go negative — and $s$ always stays in $[-1, 1]$.

In [ ]:
def silhouette_point(a, b):
    """Silhouette of one point from its a (own cluster) and b (nearest other cluster)."""
    # TODO(you): (b - a) / max(a, b)
    return ...

In [ ]:
# Checks — run me
assert abs(silhouette_point(1.0, 3.0) - 2 / 3) < 1e-12, "well-placed point: (3-1)/3"
assert silhouette_point(2.0, 2.0) == 0.0, "on the boundary between clusters"
assert silhouette_point(3.0, 1.0) < 0, "closer to the other cluster -> negative"
assert -1 <= silhouette_point(5.0, 0.5) <= 1 and -1 <= silhouette_point(0.5, 5.0) <= 1, "always in [-1, 1]"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def silhouette_point(a, b):
    return (b - a) / max(a, b)
```

</details>

### Exercise 2 — Inertia, the elbow plot's y-axis

Inertia is the within-cluster sum of squares:

$$\text{inertia} = \sum_i \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c_i} \rVert^2$$

Implement it. The checks verify a hand-computable value, that the **mean** beats any other centroid choice (the fact behind the update step), and that $k = 1$ scores far worse — the reason elbow plots slope down.

In [ ]:
def inertia(X, labels, centroids):
    """Within-cluster sum of squared distances."""
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    centroids = np.asarray(centroids, dtype=float)

    # TODO(you): each point's squared distance to ITS OWN centroid, summed
    # (hint: centroids[labels] lines up a centroid per point)
    return ...

In [ ]:
# Checks — run me
Xi = np.array([[0.0, 0.0], [2.0, 0.0], [10.0, 0.0], [12.0, 0.0]])
lab = np.array([0, 0, 1, 1])
cent = np.array([[1.0, 0.0], [11.0, 0.0]])

assert abs(inertia(Xi, lab, cent) - 4.0) < 1e-12, "four points each 1 away from their centroid: 4 * 1²"

worse = np.array([[0.0, 0.0], [11.0, 0.0]])
assert inertia(Xi, lab, cent) < inertia(Xi, lab, worse), "the mean minimizes within-cluster squared distance"

one_cluster = inertia(Xi, np.zeros(4, dtype=int), np.array([[6.0, 0.0]]))
assert one_cluster > inertia(Xi, lab, cent), "k=1 has much higher inertia — why the elbow plot drops"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def inertia(X, labels, centroids):
    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)
    centroids = np.asarray(centroids, dtype=float)
    return float(np.sum((X - centroids[labels]) ** 2))
```

</details>